<a href="https://colab.research.google.com/github/Swayam17o5/DL1/blob/main/GRU1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import random
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Input
from sklearn.metrics import classification_report
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

positive_posts = ["I had such a great day today", "Feeling really happy after meeting my friends", "This news just made my day!", "Everything is going well for once", "I am super excited about this opportunity", "Life feels good right now", "That was an amazing experience", "Finally things are working out", "I'm in such a good mood today", "Totally loving this vibe", "This project is going wonderfully", "Achieved my goals today, feeling proud!", "Had a fantastic time on my vacation", "So grateful for all the good things in my life", "Received some excellent news today", "Feeling optimistic about the future", "I love how this turned out", "Today was a success", "The sun is shining and I feel great"]
negative_posts = ["Today was honestly terrible", "I feel so stressed and tired", "Nothing is going right lately", "I am really frustrated with everything", "Worst day ever", "I feel completely drained", "This is so disappointing", "I just want this day to end", "Everything is messed up", "I am not okay today", "Having a really bad week", "This situation is making me angry", "I'm so fed up with this problem", "Feeling completely hopeless", "I hate how this is going", "This is a total disaster", "I feel so lonely and sad"]
neutral_posts = ["I went to work today", "Just finished my lunch", "It is a normal day", "Nothing much happening", "I am at home right now", "Just doing my routine tasks", "Today is a regular weekday", "I have a meeting later", "Just scrolling through my phone", "Same old day", "The sky is blue", "I need to buy groceries", "The cat is sleeping on the couch", "It's 3 o'clock in the afternoon", "I am reading a book", "The weather is quite mild", "I am sitting at my desk"]

def add_noise(text):
    noise_elements = ["!", "...", "hmm", "just", "well", "so"]
    words = text.split()
    if random.random() > 0.8:
        words.append(random.choice(noise_elements))
    return " ".join(words).strip()

data = []
for _ in range(5000):
    data.append([add_noise(random.choice(positive_posts)), "positive"])
    data.append([add_noise(random.choice(negative_posts)), "negative"])
    data.append([add_noise(random.choice(neutral_posts)), "neutral"])

df = pd.DataFrame(data, columns=["text", "sentiment"])
df = df.sample(frac=1).reset_index(drop=True)
label_map = {"negative": 0, "neutral": 1, "positive": 2}
df["label"] = df["sentiment"].map(label_map)

tokenizer = Tokenizer(num_words=2000, oov_token="<OOV>")
tokenizer.fit_on_texts(df["text"])

sequences = tokenizer.texts_to_sequences(df["text"])
X = pad_sequences(sequences, maxlen=30, padding='post')
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = Sequential([
    Input(shape=(30,)),
    Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=64),
    GRU(32),
    Dense(3, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2)
]

model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.1, callbacks=callbacks, verbose=1)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {acc:.4f}")

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=list(label_map.keys()), zero_division=0))

test_sentences = ["I am so happy today!", "This is the worst thing ever", "Just another normal day", "Feeling tired", "Life is good"]
sequences_test = tokenizer.texts_to_sequences(test_sentences)
padded_test = pad_sequences(sequences_test, maxlen=30, padding='post')

predictions = model.predict(padded_test, verbose=0)
predicted_classes = np.argmax(predictions, axis=1)
inverse_label_map = {v: k for k, v in label_map.items()}

print("\nTest Predictions:")
for sentence, cls in zip(test_sentences, predicted_classes):
    print(f"Sentence: '{sentence}' -> Sentiment: {inverse_label_map[cls]}")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 30, 64)         │         9,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,427 (75.89 KB)

 Trainable params: 19,427 (75.89 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
338/338 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.3317 - loss: 1.0997 - val_accuracy: 0.3133 - val_loss: 1.0994 - learning_rate: 0.0010
Epoch 2/20
338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.5928 - loss: 0.7154 - val_accuracy: 1.0000 - val_loss: 0.0026 - learning_rate: 0.0010
Epoch 3/20
338/338 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 1.0000 - loss: 0.0010 - val_accuracy: 1.0000 - val_loss: 5.0889e-04 - learning_rate: 0.0010
Epoch 4/20
338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 1.0000 - loss: 3.3922e-04 - val_accuracy: 1.0000 - val_loss: 2.4335e-04 - learning_rate: 0.0010
Epoch 5/20
338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 1.0000 - loss: 1.8329e-04 - val_accuracy: 1.0000 - val_loss: 1.4636e-04 - learning_rate: 0.0010
Epoch 6/20
338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 1.0000 - loss: 1.1733e-04 - val_accuracy: 1.0000 - val_loss: 9.8710e-05 - learning_rate: 0.0010
Epoch 7/20
338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accura

In [ ]:
Dataset Creation
Created 3 lists:
positive_posts, negative_posts, neutral_posts
Each contains sample sentences for each sentiment

Defined function:

add_noise(text)
Randomly adds words like "!", "hmm", "..."
Loop runs 5000 times:
Picks random sentence from each category
Adds noise
Stores in dataset

👉 Final dataset ≈ 15,000 sentences

🔹 Data Preparation

Stored data in:

pandas DataFrame

Shuffled data:

df.sample(frac=1)

Converted labels:

negative → 0
neutral → 1
positive → 2

👉 Model needs numerical labels

🔹 Tokenization (Text → Numbers)

Used:

Tokenizer(num_words=2000, oov_token="<OOV>")
Converts words into integers

Example:

"I am happy" → [5, 2, 34]
<OOV> handles unknown words
🔹 Padding Sequences

Used:

pad_sequences(maxlen=30)
Makes all sentences same length

👉 Required because neural networks need fixed-size input

🔹 Train-Test Split

Used:

train_test_split(test_size=0.2, stratify=y)
80% training, 20% testing
stratify keeps class balance
🔹 Model Architecture
Embedding → GRU → Dense
Embedding Layer
Converts word indices → dense vectors
Learns word meaning
GRU Layer
Processes sequence of words
Captures context (order matters)
Dense Layer (Softmax)
Outputs 3 probabilities:
Negative / Neutral / Positive
🔹 Model Compilation
optimizer = Adam(0.001)
loss = sparse_categorical_crossentropy
Loss used for multi-class classification
Adam optimizes learning
🔹 Training with Callbacks
EarlyStopping
Stops if validation loss stops improving
ReduceLROnPlateau
Reduces learning rate automatically

👉 Improves training efficiency

🔹 Model Training
model.fit(...)
Epochs = 20
Batch size = 32
Validation split = 10%

👉 Model learns patterns in text

🔹 Model Evaluation
model.evaluate()
classification_report()
Accuracy is printed
Classification report shows:
Precision
Recall
F1-score
🔹 Predictions on New Sentences

Input sentences:

["I am happy", "Worst day ever", ...]
Steps:
Convert to sequence
Pad
Predict

Use:

argmax()

to get final class

👉 Output example:

"I am happy" → positive